In [79]:
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from operator import add

In [80]:
load_dotenv()

True

In [81]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

In [82]:
class State(TypedDict, total=False):
    paragraph:str
    language_feedback:str
    creativity_feedback:str
    overall_score:Annotated[list[int], add]

In [83]:
class StructureFeedback(BaseModel):
    feedback:str = Field(description="give the feedback on the paragraph")
    score:int = Field(description="give the proper score on the paragraph as you see fit")

In [84]:
structured_model = llm.with_structured_output(StructureFeedback)

In [85]:
graph = StateGraph(State)

In [86]:
def language_feedback_function(state: State)->State:
    llm_output = structured_model.invoke(f"Give the language feedback on this paragraph -> {state['paragraph']}") 
    return {'language_feedback':llm_output.feedback, "overall_score":[llm_output.score]}

In [87]:
def creativity_feedback_function(state: State)->State:
    llm_output = structured_model.invoke(f"Give the creativity feedback on this paragraph -> {state['paragraph']}") 
    return {'creativity_feedback':llm_output.feedback, "overall_score":[llm_output.score]}

In [88]:
def overall_evaluation_function(state: State)->State:
    final_score = sum(state['overall_score'])
    return {'overall_score':[final_score]}

In [89]:
graph.add_node('language_feedback', language_feedback_function)
graph.add_node('creativity_feedback', creativity_feedback_function)
graph.add_node('overall_evaluation', overall_evaluation_function)

In [90]:
graph.add_edge(START, 'language_feedback')
graph.add_edge(START, 'creativity_feedback')
graph.add_edge('creativity_feedback', 'overall_evaluation')
graph.add_edge('language_feedback', 'overall_evaluation')
graph.add_edge('overall_evaluation', END)

In [91]:
workflow = graph.compile()

In [92]:
input_para = {
    "paragraph": "once upon a time there was a lion who was brave and strong but he died in a battle with two evil lions while protection his son but later his son when he became adult went back to take the revenge of his father and killed both of them and finally took revenge and the territory back from those bastards."
}

In [93]:
workflow.invoke(input_para)

{'paragraph': 'once upon a time there was a lion who was brave and strong but he died in a battle with two evil lions while protection his son but later his son when he became adult went back to take the revenge of his father and killed both of them and finally took revenge and the territory back from those bastards.',
 'language_feedback': 'This paragraph has several language and structural issues that need improvement:\n\n1. **Grammar and Tense Consistency**: "while protection his son" should be "while protecting his son." The gerund form is needed here.\n\n2. **Run-on Sentence**: The entire paragraph is one long, rambling sentence. It should be broken into multiple sentences for clarity and better flow. For example: "Once upon a time, there was a brave and strong lion. He died in a battle with two evil lions while protecting his son. Later, when his son became an adult..."\n\n3. **Word Choice**: "when he became adult" should be "when he became an adult" (article needed).\n\n4. **Awk